In [1]:
!pip install rasterio geopandas ultralytics optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 24.7 MB/s eta 0:00:00


In [2]:
import cv2
import time
import yaml
import json
import torch
import optuna
import pickle
import logging
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Dict

import albumentations as A
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict
from albumentations.pytorch import ToTensorV2
from typing import Dict, List, Tuple, Optional

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
def get_gpu_info():
    """Check available GPU resources in Kaggle"""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {gpu_name}")
        print(f"GPU Memory: {gpu_memory:.1f} GB")
        return gpu_memory
    else:
        print("No GPU available, using CPU")
        return 0

In [6]:
def train_progressive_augmentation(model, project, name):
    """
    Progressive training with two phases
    Phase 1: Initial training (60 epochs)
    Phase 2: Fine-tuning with lower learning rate (40 epochs)
    """
    from ultralytics import YOLO

    gpu_memory = get_gpu_info()

    # Base configuration
    base_config = {
        'data': '/content/drive/MyDrive/AGRI/Detection/dataset/data.yaml',
        'imgsz': 640,
        'batch': 8,
        'task': 'detect',
        'project': project,
        'optimizer': 'AdamW',
        'amp': True,
        'device': 0,
        'workers': 2,
        'cache': False,
        'cos_lr': True,
        'val': True,
        'plots': True,
        'save': True,
        'verbose': True,
        'seed': 42,
        'exist_ok': True,
        'pretrained': True,
        'patience': 0,
    }

    # Phase 1: Initial training (60 epochs)
    print("\n🚀 Phase 1: Initial Training (60 epochs)")
    phase1_config = base_config.copy()
    phase1_config.update({
        'name': f'{name}_phase1',
        'epochs': 60,
        'lr0': 0.001,
    })

    results_phase1 = model.train(**phase1_config)

    # Get the path to the best model from phase 1
    best_model_path = results_phase1.save_dir / 'weights' / 'best.pt'
    print(f"\n✅ Phase 1 complete! Best model: {best_model_path}")

    # Phase 2: Load the best model from Phase 1 and fine-tune
    print("\n🎯 Phase 2: Fine-tuning with Light Augmentation (40 epochs)")

    # Load the best model from Phase 1
    model_phase2 = YOLO(str(best_model_path))

    phase2_config = base_config.copy()
    phase2_config.update({
        'name': f'{name}_phase2',
        'epochs': 40,
        'lr0': 0.0001,  # Lower learning rate for fine-tuning
        'mosaic': 0.3,  # Reduced mosaic
        'mixup': 0.1,   # Minimal mixup
        'copy_paste': 0.1,
        'degrees': 5.0,  # Reduced rotation
        'translate': 0.05,
        'scale': 0.3,
        'hsv_s': 0.4,
        'hsv_v': 0.3,
        'erasing': 0.2,
        'close_mosaic': 5,
        'resume': False,  # Start fresh training, not resume
    })

    results_phase2 = model_phase2.train(**phase2_config)

    print("✅ Progressive training completed!")
    print(f"Final model: {results_phase2.save_dir / 'weights' / 'best.pt'}")

    return results_phase2

In [8]:
# Usage examples:
def main():
    """Example usage in Kaggle"""

    # Load your model
    model = YOLO('yolo11l.pt')  # or your specific model
    project = '/content/drive/MyDrive/AGRI/Detection/tree_detection'
    name = 'augmented_run_v3'

    print("Choose training method:")
    print("1. Standard training with heavy augmentation")
    print("2. Progressive training (recommended for best results)")

    # Option 1: Standard training with comprehensive augmentation
    # results = train_with_augmentation(model, project, name)

    # Option 2: Progressive training (uncomment to use instead)
    results = train_progressive_augmentation(model, project, name)

    # After training, you can evaluate or make predictions
    print(f"Training completed! Best model saved in: {project}/{name}/weights/best.pt")

    return results

if __name__ == "__main__":
    main()

Choose training method:
1. Standard training with heavy augmentation
2. Progressive training (recommended for best results)
GPU: Tesla T4
GPU Memory: 15.8 GB

🚀 Phase 1: Initial Training (60 epochs)
Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/AGRI/Detection/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0,